# Instructions

In this tutorial, we will perform multi-label classification using an ECG-FM model finetuned on the [MIMIC-IV-ECG v1.0 dataset](https://physionet.org/content/mimic-iv-ecg/1.0/). It outlines the data and model loading, as well as inference, same-sample prediction aggregation, and visualizations for embeddings and saliency maps.

ECG-FM was developed in collaboration with the [fairseq_signals](https://github.com/Jwoo5/fairseq-signals) framework, which implements a collection of deep learning methods for ECG analysis.

This is segment the ECG into inputs of 5 s and perform a label-specific aggregation of the predictions from each sample

This document serves largely as a quickstart introduction. Much of this functionality is also available via the [fairseq-signals scripts](https://github.com/bowang-lab/ECG-FM/blob/main/notebooks/infer_cli.ipynb), as well the [ECG-FM scripts](https://github.com/bowang-lab/ECG-FM/tree/main/scripts).

## Installation

Begin by cloning [fairseq_signals](https://github.com/Jwoo5/fairseq-signals) and refer to the installation section in the top-level README. For example, the following commands are sufficient at the present moment:
```
# Creating `fairseq` environment:
conda create --name fairseq python=3.10.6
source activate fairseq
git clone https://github.com/Jwoo5/fairseq-signals
cd fairseq-signals
python3 -m pip install pip==24.0
python3 -m pip install -e .
```

In [1]:
import os

root = os.path.dirname(os.getcwd())

## Download checkpoints

Checkpoints are available on [HuggingFace](https://huggingface.co/wanglab/ecg-fm). The finetuned model be downloaded using the following command:

In [2]:
import sys
sys.executable
import torch, transformers, numpy as np
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("numpy", np.__version__)

/home/sagemaker-user/.conda/envs/fairseq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cu128
transformers 4.30.2
numpy 2.2.6


In [3]:
import os
from huggingface_hub import hf_hub_download

_ = hf_hub_download(
    repo_id='wanglab/ecg-fm',
    filename='mimic_iv_ecg_finetuned.yaml',
    local_dir=os.path.join(root, 'ckpts'),
)

# Inference

In [4]:
ckpt_path: str = os.path.join(root, 'ckpts/mimic_iv_ecg_finetuned.pt')
assert os.path.isfile(ckpt_path)

device: str = 'cuda'
batch_size: int = 16
num_workers: int = 0

extract_saliency: bool = True

In [5]:
import os
import pandas as pd


from typing import Any, List

def to_list(obj: Any) -> List[Any]:
    if isinstance(obj, list):
        return obj

    if isinstance(obj, (np.ndarray, set, dict)):
        return list(obj)

    return [obj]


CSV_PATH = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/mimic_iv_ecg_test10k_afib.csv"
WFDB_ROOT = "/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files"

df = pd.read_csv(CSV_PATH)

file_paths = [
    os.path.join(
        WFDB_ROOT,
        str(sp).replace("files/", "", 1)
    )
    for sp in df["source_path"]
]

file_paths[:5]

['/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1051/p10512468/s42341564/42341564',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1721/p17211916/s40965333/40965333',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1832/p18323186/s44766763/44766763',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1002/p10022863/s40441658/40441658',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1720/p17203862/s41908718/41908718']

## Prepare data

To simplify this tutorial, we have processed a sample of 10 ECGs (14 5s segments) from the [CODE-15% v1.0.0 dataset](https://zenodo.org/records/4916206/) using our [end-to-end data preprocessing pipeline](https://github.com/Jwoo5/fairseq-signals/tree/master/scripts/preprocess/ecg). Its README is also helpful if looking to perform inference using your own dataset, where there are already preprocessing scripts implemented for several public datasets.

In [6]:
from typing import List
from itertools import chain

from scipy.io import loadmat

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader

from ecg_transform.inp import ECGInput, ECGInputSchema
from ecg_transform.sample import ECGMetadata, ECGSample
from ecg_transform.t.base import ECGTransform
from ecg_transform.t.common import (
    HandleConstantLeads,
    LinearResample,
    ReorderLeads,
)
from ecg_transform.t.scale import Standardize
from ecg_transform.t.cut import SegmentNonoverlapping

class ECGFMDataset(Dataset):
    def __init__(
        self,
        schema,
        transforms,
        file_paths,
    ):
        self.schema = schema
        self.transforms = transforms
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        import wfdb
        import numpy as np

        # WFDB record prefix path, e.g. .../s41420867/41420867
        record_path = self.file_paths[idx]

        record = wfdb.rdrecord(record_path)
        feats = record.p_signal.T  # (leads, time)
        org_sample_rate = int(record.fs)

        metadata = ECGMetadata(
            sample_rate=org_sample_rate,
            num_samples=feats.shape[1],
            lead_names=['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6'],
            unit=None,
            input_start=0,
            input_end=feats.shape[1],
        )
        metadata.file = record_path
        inp = ECGInput(feats, metadata)
        sample = ECGSample(
            inp,
            self.schema,
            self.transforms,
        )
        source = torch.from_numpy(sample.out).float()

        return source, inp

def collate_fn(inps):
    sample_ids = list(
        chain.from_iterable([[inp[1]] * inp[0].shape[0] for inp in inps])
    )
    return torch.concatenate([inp[0] for inp in inps]), sample_ids

def file_paths_to_loader(
    file_paths: List[str],
    schema: ECGInputSchema,
    transforms: List[ECGTransform],
    batch_size=1,
    num_workers=0,
):
    dataset = ECGFMDataset(
        schema,
        transforms,
        file_paths,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        sampler=None,
        shuffle=False,
        collate_fn=collate_fn,
        drop_last=False,
    )

In [7]:
ECG_FM_LEAD_ORDER = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SAMPLE_RATE = 500
N_SAMPLES = SAMPLE_RATE*5

label_def = pd.read_csv(
    os.path.join(root, 'data/mimic_iv_ecg/labels/label_def.csv'),
     index_col='name',
)
label_names = label_def.index.to_list()
label_names

['Poor data quality',
 'Sinus rhythm',
 'Premature ventricular contraction',
 'Tachycardia',
 'Ventricular tachycardia',
 'Supraventricular tachycardia with aberrancy',
 'Atrial fibrillation',
 'Atrial flutter',
 'Bradycardia',
 'Accessory pathway conduction',
 'Atrioventricular block',
 '1st degree atrioventricular block',
 'Bifascicular block',
 'Right bundle branch block',
 'Left bundle branch block',
 'Infarction',
 'Electronic pacemaker']

In [8]:
AGG_METHODS = {
    'Poor data quality': 'max',
    'Sinus rhythm': 'mean',
    'Premature ventricular contraction': 'max',
    'Tachycardia': 'mean',
    'Ventricular tachycardia': 'max',
    'Supraventricular tachycardia with aberrancy': 'max',
    'Bradycardia': 'mean',
    'Infarction': 'mean',
    'Atrioventricular block': 'mean',
    'Right bundle branch block': 'mean',
    'Left bundle branch block': 'mean',
    'Electronic pacemaker': 'max',
    'Atrial fibrillation': 'mean',
    'Atrial flutter': 'mean',
    'Accessory pathway conduction': 'mean',
    '1st degree atrioventricular block': 'mean',
    'Bifascicular block': 'mean',
}

ECG_FM_SCHEMA = ECGInputSchema(
    sample_rate=SAMPLE_RATE,
    expected_lead_order=ECG_FM_LEAD_ORDER,
    required_num_samples=N_SAMPLES,
)

ECG_FM_TRANSFORMS = [
    ReorderLeads(
        expected_order=ECG_FM_LEAD_ORDER,
        missing_lead_strategy='raise',
    ),
    LinearResample(desired_sample_rate=SAMPLE_RATE),
    HandleConstantLeads(strategy='zero'),
    Standardize(),
    SegmentNonoverlapping(segment_length=N_SAMPLES),
]

loader = file_paths_to_loader(
    file_paths,
    ECG_FM_SCHEMA,
    ECG_FM_TRANSFORMS,
    batch_size=batch_size,
    num_workers=num_workers,
)

## Load model

In [9]:
from typing import Dict, List, Optional, Tuple, Type, Union
from collections import OrderedDict

import numpy as np
import pandas as pd

import torch

from fairseq_signals.models import build_model_from_checkpoint
from fairseq_signals.models.classification.ecg_transformer_classifier import (
    ECGTransformerClassificationModel
)

In [10]:
model: ECGTransformerClassificationModel = build_model_from_checkpoint(
    checkpoint_path=ckpt_path
)

# Forcibly enable the return of attention weights for saliency maps
if extract_saliency:
    model.encoder.encoder.need_weights = extract_saliency
    for layer in model.encoder.encoder.layers:
        layer.need_weights = extract_saliency

model.eval()
model.to(device)

ECGTransformerClassificationModel(
  (encoder): ECGTransformerModel(
    (dropout_input): Dropout(p=0.0, inplace=False)
    (dropout_features): Dropout(p=0.0, inplace=False)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-11): 12 x TransformerEncoderLayer(
          (self_attn): MultiHeadAttention(
            (dropout): Dropout()
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (dropout1): Dropout(p=0.0, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
          (dropout3): Dropout(p=0.0, inplace=False)
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (f

## Infer

In [11]:
def encoder_out_to_emb(x, device='cpu'):
    # fairseq_signals/models/classification/ecg_transformer_classifier.py
    return torch.div(x.sum(dim=1), (x != 0).sum(dim=1))

def infer(
    model,
    loader,
    device,
    extract_saliency: bool = False,
):
    model.eval()

    inps = []
    sources = []
    logits = []
    embs = []
    saliency = []
    file_names = []

    with torch.no_grad():
        for source, inp in loader:
            source = source.to(device)
            out = model(source=source)
            inps.extend(inp)
            sources.append(source.detach().cpu())
            logits.append(out['out'].detach().cpu())
            embs.append(encoder_out_to_emb(out['encoder_out']).detach().cpu())
            if extract_saliency:
                saliency.append(out['saliency'].detach().cpu())
            file_names.extend([i.meta.file for i in inp])

            del source, out
            torch.cuda.empty_cache()

    # Handle predictions
    pred = torch.sigmoid(torch.concatenate(logits)).numpy()
    pred = pd.DataFrame(pred, columns=label_names, index=file_names)

    results = {
        'inps': inps,
        'sources': torch.concatenate(sources).numpy(),
        'embs': torch.concatenate(embs).numpy(),
        'pred': pred,
    }

    # Handle saliency
    if extract_saliency:
        saliency = torch.concatenate(saliency)
        attn = saliency[:, -1] # Consider only the last attention layer
        results['attn_max'] = attn.max(axis=2).values.squeeze().numpy()

    return results

In [12]:
results = infer(model, loader, device, extract_saliency=False)

In [13]:
pred = results['pred']
print(f"Number of 5 s segment predictions: {len(pred)}.")
pred

Number of 5 s segment predictions: 20000.


,Poor data quality,Sinus rhythm,Premature ventricular contraction,Tachycardia,Ventricular tachycardia,Supraventricular tachycardia with aberrancy,Atrial fibrillation,Atrial flutter,Bradycardia,Accessory pathway conduction,Atrioventricular block,1st degree atrioventricular block,Bifascicular block,Right bundle branch block,Left bundle branch block,Infarction,Electronic pacemaker
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1051/p10512468/s42341564/42341564,0.344258,0.972424,0.323295,0.037375,1.243514e-08,0.007826,0.023537,0.025777,0.043318,0.039878,0.014182,0.005687,0.000026,0.001007,0.000440,0.291147,0.017738
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1051/p10512468/s42341564/42341564,0.123265,0.964068,0.841627,0.117335,1.084392e-06,0.095699,0.061724,0.392881,0.045296,0.131519,0.014781,0.007634,0.000160,0.001319,0.000378,0.594856,0.012282
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1721/p17211916/s40965333/40965333,0.065284,0.035121,0.777906,0.934033,7.311765e-04,0.554294,0.845374,0.922856,0.112551,0.951297,0.393958,0.559976,0.018757,0.002311,0.018358,0.017746,0.462904
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1721/p17211916/s40965333/40965333,0.085932,0.009137,0.889201,0.986808,4.699413e-04,0.781689,0.973280,0.843757,0.056700,0.994128,0.151963,0.104817,0.054561,0.005376,0.006347,0.014036,0.395871
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1832/p18323186/s44766763/44766763,0.044855,0.010294,0.921988,0.998754,1.771273e-07,0.970730,0.994416,0.172365,0.002833,0.998954,0.004484,0.001696,0.000094,0.019302,0.014715,0.961118,0.042385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1938/p19385108/s42439567/42439567,0.041692,0.337293,0.566482,0.866430,1.048353e-11,0.965645,0.475187,0.039637,0.018273,0.479087,0.977528,0.977760,0.161089,0.983960,0.000057,0.449522,0.808155
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1884/p18849711/s45571703/45571703,0.583222,0.062160,0.955533,0.971192,2.812943e-04,0.989295,0.972819,0.019340,0.157802,0.940091,0.025186,0.029682,0.962280,0.993084,0.000899,0.986023,0.121980
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1884/p18849711/s45571703/45571703,0.645824,0.067474,0.941645,0.959458,6.123146e-06,0.993307,0.968438,0.053504,0.497227,0.940365,0.083915,0.110830,0.960367,0.994293,0.002488,0.984683,0.422886
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1779/p17797856/s49541450/49541450,0.217551,0.998235,0.192918,0.991725,3.085626e-07,0.006058,0.000721,0.000006,0.010353,0.000258,0.000346,0.000110,0.000004,0.000127,0.000106,0.862014,0.004938


# Result handling

## Prediction aggregation

In [14]:
pred_agg = pred.groupby(pred.index).agg(AGG_METHODS).astype(float)
print(f"Number of sample-aggregated predictions: {len(pred_agg)}.")
pred_agg

Number of sample-aggregated predictions: 10000.


,Poor data quality,Sinus rhythm,Premature ventricular contraction,Tachycardia,Ventricular tachycardia,Supraventricular tachycardia with aberrancy,Bradycardia,Infarction,Atrioventricular block,Right bundle branch block,Left bundle branch block,Electronic pacemaker,Atrial fibrillation,Atrial flutter,Accessory pathway conduction,1st degree atrioventricular block,Bifascicular block
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10000635/s43522917/43522917,0.060183,0.997866,0.952732,0.002723,2.437005e-08,0.006227,0.995087,0.029409,0.003160,0.002551,0.002688,0.055664,0.006493,0.000166,0.003908,0.000168,0.000832
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10000635/s48339811/48339811,0.060072,0.970849,0.758486,0.025581,2.917625e-08,0.059251,0.974054,0.010329,0.007906,0.024241,0.002951,0.053032,0.059806,0.011708,0.054897,0.002125,0.001915
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10002559/s41675681/41675681,0.182869,0.996500,0.627020,0.006384,1.116430e-12,0.006989,0.001029,0.003397,0.002913,0.453928,0.000153,0.000907,0.000890,0.000005,0.000770,0.001414,0.000030
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10003019/s44989970/44989970,0.260792,0.557203,0.928344,0.376003,1.382516e-07,0.653466,0.028995,0.196265,0.204115,0.052561,0.005632,0.098687,0.356966,0.806047,0.581821,0.062334,0.105671
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10003019/s46272123/46272123,0.014521,0.845018,0.946704,0.029090,1.927414e-06,0.036456,0.126351,0.036775,0.005584,0.331765,0.003036,0.074875,0.032960,0.032133,0.023812,0.001758,0.681571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1998/p19989126/s41616922/41616922,0.124507,0.998830,0.223216,0.005866,5.219356e-08,0.000139,0.990655,0.051788,0.000653,0.000534,0.000424,0.031259,0.002658,0.000007,0.001145,0.000363,0.000010
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1998/p19989183/s49667577/49667577,0.040723,0.919465,0.169322,0.035146,1.039498e-09,0.000779,0.008032,0.004663,0.021042,0.000274,0.001735,0.007854,0.006767,0.005617,0.011262,0.017697,0.000076
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1999/p19993776/s45691497/45691497,0.138209,0.144829,0.154726,0.024271,2.449882e-08,0.033831,0.056750,0.088094,0.042208,0.002697,0.584769,0.876225,0.062421,0.007690,0.046112,0.085698,0.000947
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1999/p19996783/s45159013/45159013,0.107795,0.857151,0.482036,0.978326,1.069305e-08,0.029361,0.013277,0.997283,0.099857,0.000187,0.000989,0.027405,0.024875,0.427218,0.190772,0.031652,0.000001


In [17]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

CSV_PATH = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/mimic_iv_ecg_test10k_afib.csv"
WFDB_ROOT = "/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files"

# 读你自己的 CSV
df_gt = pd.read_csv(CSV_PATH)

# 构造和 pred_agg.index 一致的 record path
df_gt["file"] = df_gt["source_path"].apply(
    lambda sp: os.path.join(WFDB_ROOT, str(sp).replace("files/", "", 1))
)

# 第5列是 afib 标签（Python 0-based 下就是 iloc[:, 4]）
df_gt["afib"] = df_gt.iloc[:, 4].astype(int)

# 设成索引，方便和 pred_agg 对齐
df_gt = df_gt.set_index("file")

# 只评估 AFib
pred_col = "Atrial fibrillation"
common = pred_agg.index.intersection(df_gt.index)

print("Evaluation samples:", len(common))

if len(common) == 0:
    print("No overlapping samples between pred_agg.index and CSV ground truth index.")
    print("\npred_agg sample index:")
    print(pred_agg.index[:5].tolist())
    print("\nground truth sample index:")
    print(df_gt.index[:5].tolist())
else:
    y_true = df_gt.loc[common, "afib"].values
    y_prob = pred_agg.loc[common, pred_col].values
    y_pred = (y_prob >= 0.5).astype(int)

    accuracy = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print("Evaluated label:", pred_col)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1:       {f1:.4f}")

    if len(np.unique(y_true)) >= 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
        print(f"AUROC:    {auroc:.4f}")
        print(f"AUPRC:    {auprc:.4f}")
    else:
        print("AUROC/AUPRC skipped because y_true has only one class.")

Evaluation samples: 10000
Evaluated label: Atrial fibrillation
Accuracy: 0.9507
F1:       0.8019
AUROC:    0.9917
AUPRC:    0.9383


## Visualizing embeddings

In [18]:
source, inp = next(iter(loader))

print("Source shape:", source.shape)
print("Lead energy:", torch.sum(torch.abs(source), dim=2)[0])

Source shape: torch.Size([32, 12, 2500])
Lead energy: tensor([1911.3196, 1913.4805, 2123.2754, 1872.6987, 2249.6455, 1951.6646,
        1974.1024, 1947.7042, 1665.5200, 1560.5485, 1482.9775, 1432.6455])
